### IMPORTAÇÕES

In [2]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd
import datetime
import awswrangler as awr
import openpyxl
import os
import shutil
import datetime as dt
import pyautogui
import time
import matplotlib.pyplot as plt
import matplotlib
from html2image import Html2Image
matplotlib.use('Agg')  

In [3]:
# FORMATANDO EM CAIXA ALTA
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

### EXTRAÇÕES DE BASES

In [ ]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos = awr.athena.read_sql_query(
    sql=sql_ativos,
    database='silver',
    ctas_approach=False
)

In [5]:
# ELIMINANDO DUPLICATAS DE ATIVOS POR CHASSI

df_ativos = df_ativos.drop(columns=['rn'])

df_ativos = (
    df_ativos
    .drop_duplicates(subset=['chassi'], keep='first')
)

df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [6]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver',
    ctas_approach=False
)

In [7]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver',
    ctas_approach=False
)

In [8]:
# FORMATAÇÃO EM CAIXA ALTA
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,ARY6C38,9AA07072CAC088215,3348,0,3348,4293,15929,UNIDADE ITAPEJARA,DENILSO BRANCALIONE,ATIVO,...,<NA>,98675,CASCO (R/SR),CANCELADO,2026-05-21,2025-09-24,2025-10-09,2025-10-09,2026-05-14,STCOOP
1,NVG9C26,9BSR6X200B3677189,11110,11110,<NA>,21085,15299,UNIDADE CASCAVEL,MAURICIO BELLO,ATIVO,...,<NA>,94977,REPARAÇÃO A TERCEIROS,CANCELADO,2026-05-21,2025-07-28,2025-07-30,2025-07-30,2026-04-10,STCOOP
2,RRW6H58,9ADM0352PPM526413,17989,0,17989,21912,15094,UNIDADE VILHENA,ISABELLY MELLO,ATIVO,...,<NA>,93739,CASCO (R/SR),CANCELADO,2026-05-21,2025-07-08,2025-07-17,2025-07-17,2026-04-28,STCOOP
3,DPF4E88,9BW9J82477R701276,13865,13865,<NA>,11915,16420,MF - MARTUCCI SOLUCOES,AMILTON MARTUCCI,ATIVO,...,<NA>,101543,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-05-21,2025-11-21,2025-12-26,2025-12-29,2026-05-19,STCOOP
4,DJB2996,93KK6AAC44E100045,12208,12208,<NA>,11915,16420,MF - MARTUCCI SOLUCOES,AMILTON MARTUCCI,ATIVO,...,<NA>,101520,CASCO (VEÍCULO),CANCELADO,2026-05-21,2025-11-21,2025-12-26,2025-12-29,2026-05-19,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026,SEG0F44,9APM06220PP000756,28741,0,28741,493,6698,MICRO B - SAMUEL MARTINS,DAIANE CRISTINA VEIGA DA SILVA,ATIVO,...,<NA>,748746,CASCO PT (R/SR),CANCELADO,2026-05-21,2025-11-04,2025-11-10,2025-11-10,2026-05-13,TAG
2027,SEA0J98,9APB08820NP007078,28533,0,28533,493,6698,MICRO B - SAMUEL MARTINS,DAIANE CRISTINA VEIGA DA SILVA,ATIVO,...,<NA>,748741,ASSISTÊNCIA A REPAROS (R/SR),CANCELADO,2026-05-21,2025-11-04,2025-11-10,2025-11-10,2026-05-13,TAG
2028,SEA1A47,9APM06220NP000676,28529,0,28529,493,6698,MICRO B - SAMUEL MARTINS,DAIANE CRISTINA VEIGA DA SILVA,ATIVO,...,<NA>,748741,ASSISTÊNCIA A REPAROS (R/SR),CANCELADO,2026-05-21,2025-11-04,2025-11-10,2025-11-10,2026-05-13,TAG
2029,SEA0J98,9APB08820NP007078,28533,0,28533,493,6698,MICRO B - SAMUEL MARTINS,DAIANE CRISTINA VEIGA DA SILVA,ATIVO,...,<NA>,748740,CASCO PT (R/SR),CANCELADO,2026-05-21,2025-11-04,2025-11-10,2025-11-10,2026-05-13,TAG


In [9]:
# DEFININDO TODAY TS e YESTERDAY

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')

yesterday

'20-05-2026'

### JUNÇÃO DE BASES DE CANCELAMENTO

In [10]:
# ATUALIZANDO NOME COLUNA DE ATUALIZAÇÃO DE CANCELAMENTOS PARCIAIS

df_cancelamentos_parciais = df_cancelamentos_parciais.rename(columns={'data_atualizacao': 'data_cancelamento'})

In [11]:
# CRIANDO COLUNAS DE IDENTIFICADOR DE CANCELAMENTO

df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [12]:
# RETIRANDO DE CANCELAMENTOS PARCIAIS E DE CANCELAMENTOS INTEGRAIS AS PLACAS ATIVAS (PODEM TER SIDO RENOVADAS EM OUTRO CONJUNTO)

lista_ativos = df_ativos['chassi'].to_list()
df_cancelamentos_parciais = df_cancelamentos_parciais[~df_cancelamentos_parciais['chassi'].isin(lista_ativos)]


In [13]:
# CONCATENANDO BASES DE CANCELAMENTO E RETIRANDO DUPLICATAS POR CHASSI

df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

# Converter data_cancelamento para Timestamp antes de sort_values para evitar erro de comparação
df_cancelamentos['data_cancelamento'] = pd.to_datetime(
    df_cancelamentos['data_cancelamento'], errors='coerce'
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos = df_cancelamentos.drop_duplicates(subset=['chassi'], keep='first')

In [14]:
# ACRESCENTANDO COLUNA DE FRANQUEAMENTO OU NÃO

def map_franqueado(tem_palavra_chave):
    # Se for NA (pandas.NA), trata como "SIM"
    if pd.isna(tem_palavra_chave):
        return "SIM"
    return "UNIDADES/FRANQUIAS" if tem_palavra_chave else "PARCEIROS/CORRETORES/FTR"

df_cancelamentos['categoria_comercial'] = df_cancelamentos['unidade'].str.contains(
    "UNIDADE|MF|FRANQUEADO|MICRO|TS CONSULTORIA|TRANSDESK DIGITAL", na=False, case=False
).apply(map_franqueado)

In [15]:
# FORMATANDO COLUNAS DE DATA DE CANCELAMENTO PARA DT (TS)

df_cancelamentos['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos['data_cancelamento'], errors='coerce'
).dt.date

In [16]:
# DEFININDO TODAY e YESTERDAY

today = pd.Timestamp.today().date()

yesterday = today - dt.timedelta(days=1)
sexta = yesterday - dt.timedelta(days=2)
sabado = yesterday - dt.timedelta(days=1)

### ACRESCENTANDO VISUALIZAÇÃO DE INADIMPLÊNCIA

In [17]:
#CRIANDO DATAFRAME DE INADIMPLÊNCIA 45+
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_inadimplencia\sql\faturas_45+.sql"
with open(caminho_query, 'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas = awr.athena.read_sql_query(query, database='silver')

boletos_pagos = df_faturas.loc[df_faturas['historico'] != 1, 'ponteiro'].tolist()
df_inadimp = df_faturas[~df_faturas['ponteiro'].isin(boletos_pagos)]
df_inadimp = df_inadimp[df_inadimp['historico'] == 1]
df_inadimp = df_inadimp.drop_duplicates(subset=['ponteiro', 'conjunto', 'matricula', 'empresa'], keep='first')
# a pedido do setor de COBRANÇAS, retirar associado: APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D
# a pedido do setor de COBRANÇAS, retirar associado: TESTE
df_inadimp_atual_45 = df_inadimp[
    ~(
        (df_inadimp['empresa'].isin(['Viavante', 'Stcoop', 'Segtruck'])) &
        (df_inadimp['associado'] == "APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D")
    )
]
df_inadimp_atual_45 = df_inadimp_atual_45[~df_inadimp_atual_45['associado'].str.contains('TESTE', na=False)]

#caminho_pasta = r'C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel\Relatório de Inadimplência'
#caminho_arquivo = os.path.join(caminho_pasta, 'relatorio_inadimplencia.xlsx')
#os.makedirs(caminho_pasta, exist_ok=True)
#if os.path.exists(caminho_arquivo):
#    os.remove(caminho_arquivo)
#    print("Arquivo antigo removido, iniciando carregamento...")
#df_inadimp_atual_45.to_excel(caminho_arquivo, engine='openpyxl', index=False, sheet_name='inadimplentes')
#print(f"Arquivo Excel salvo com sucesso em: {caminho_arquivo}")

In [18]:
# CONCATENANDO COM A BASE DE CANCELAMENTOS

df_cancelamentos['inadimplente_45+'] = df_cancelamentos['conjunto'].isin(df_inadimp_atual_45['conjunto']).map({True: 'SIM', False: 'NÃO'})

### CONSTRUINDO EXCEL

In [19]:
# DEFININDO CÉLULAS RELATÓRIO w4
df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

if today.weekday() == 0: 
    ativos_mask = (df_ativos['inicio_vig'] >= sexta) & (df_ativos['inicio_vig'] <= yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] >= sexta) & (df_cancelamentos['data_cancelamento'] <= yesterday)
else:
    ativos_mask = (df_ativos['inicio_vig'] == yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] == yesterday)

ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & ativos_mask])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & ativos_mask])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & ativos_mask])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & ativos_mask])
total_ativados = len(df_ativos[ativos_mask])


In [20]:
#FILTRANDO CÉLULAS RELATÓRIO w4

#DEFININDO NOVOS FILTROS
parciais_mask = df_cancelamentos['identificador'] == 'PARCIAL'
integrais_mask = df_cancelamentos['identificador'] == 'INTEGRAL'
cancelados_mask = df_cancelamentos['status'] == 'CANCELADO'
finalizados_mask = df_cancelamentos['status'] == 'FINALIZADO'

#DIVIDINDO RESULTADOS
cancelados_seg_parc = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask & parciais_mask])
cancelados_st_parc = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask & parciais_mask])
cancelados_viav_parc = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask & parciais_mask])
cancelados_tag_parc = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask & parciais_mask])

cancelados_seg_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask & integrais_mask & cancelados_mask])
cancelados_st_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask & integrais_mask & cancelados_mask])
cancelados_viav_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask & integrais_mask & cancelados_mask])
cancelados_tag_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask & integrais_mask & cancelados_mask])

finalizados_seg_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask & integrais_mask & finalizados_mask])
finalizados_st_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask & integrais_mask & finalizados_mask])
finalizados_viav_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask & integrais_mask & finalizados_mask])
finalizados_tag_int = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask & integrais_mask & finalizados_mask])
total_cancelados = len(df_cancelamentos[cancelamentos_mask])

In [21]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [22]:
# DEFININDO NOMES DAS ABAS E FAZENDO A LIMPEZA DE w1 e w2

template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']
w5 = wb['CANCELAMENTOS ONTEM']


clear_sheet(w1)
clear_sheet(w2)
clear_sheet(w5)

In [23]:
# FORMATANDO LINHAS NULAS

numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')


W1 - ATIVOS

In [24]:

for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

W2 - CANCELADOS

In [25]:

for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

W5 - CANCELADOS ONTEM

In [26]:
#FILTRANDO DF_CANCELAMENTOS
df_cancelamentos_ontem = df_cancelamentos[cancelamentos_mask] 

#FORMATANDO LINHAS NULAS
numeric_cols = df_cancelamentos_ontem.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos_ontem.select_dtypes(include=['object', 'string']).columns

df_cancelamentos_ontem[numeric_cols] = df_cancelamentos_ontem[numeric_cols].fillna(0)
df_cancelamentos_ontem[string_cols] = df_cancelamentos_ontem[string_cols].fillna('N/A')

for c_idx, col_name in enumerate(df_cancelamentos_ontem.columns, start=1):
    w5.cell(row=1, column=c_idx, value=col_name)

if  df_cancelamentos_ontem.empty == False:
    for r_idx, row in enumerate(df_cancelamentos_ontem.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w5.cell(row=r_idx, column=c_idx, value=value)

C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_47588\2413304320.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cancelamentos_ontem[numeric_cols] = df_cancelamentos_ontem[numeric_cols].fillna(0)
C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_47588\2413304320.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cancelamentos_ontem[string_cols] = df_cancelamentos_ontem[string_cols].fillna('N/A')


W3 - BASE

In [27]:
first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value.date()


In [28]:
# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM  
#if data_atual_planilha == yesterday:
    # Filtra ativos até a data de ontem (inclusive)
if "inicio_vig" in df_ativos.columns:
    # Supondo que data_ativacao já seja datetime ou string convertível
    data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')
    df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
    qtd_ativos = df_ativos_filtrados['chassi'].nunique()
else:
    qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos

#definindo ativados totais

data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
qtd_ativos_dom = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sabado)]
qtd_ativos_sab = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sexta)]
qtd_ativos_sex = df_ativos_filtrados['chassi'].nunique()


#definindo ativações de sexta e sábado
ativos_mask_dom = df_ativos['inicio_vig'] == yesterday
total_ativados_dom = len(df_ativos[ativos_mask_dom])

ativos_mask_sab = df_ativos['inicio_vig'] == sabado
total_ativados_sab = len(df_ativos[ativos_mask_sab])

ativos_mask_sex = df_ativos['inicio_vig'] == sexta
total_ativados_sex= len(df_ativos[ativos_mask_sex])

#definindo cancelamentos de sexta e sábado
cancel_mask_dom = df_cancelamentos['data_cancelamento'] == yesterday
total_cancel_dom = len(df_cancelamentos[cancel_mask_dom])

cancel_mask_sab = df_cancelamentos['data_cancelamento'] == sabado
total_cancel_sab = len(df_cancelamentos[cancel_mask_sab])

cancel_mask_sex = df_cancelamentos['data_cancelamento'] == sexta
total_cancel_sex = len(df_cancelamentos[cancel_mask_sex])

#hoje = dt.date.today()
if today.weekday() == 0 and first_empty_row >= 3:
        w3['B' + str(first_empty_row + 2)] = qtd_ativos_dom
        w3['C' + str(first_empty_row + 2)] = total_ativados_dom
        w3['D' + str(first_empty_row + 2)] = total_cancel_dom

        w3['B' + str(first_empty_row + 1)] = qtd_ativos_sab
        w3['C' + str(first_empty_row + 1)] = total_ativados_sab
        w3['D' + str(first_empty_row + 1)] = total_cancel_sab

        w3['B' + str(first_empty_row)] = qtd_ativos_sex
        w3['C' + str(first_empty_row)] = total_ativados_sex
        w3['D' + str(first_empty_row)] = total_cancel_sex

        print('Registro de ativos preenchido para domingo, sábado e ativos na aba BASE!')
else:
        w3['B' + str(first_empty_row)] = qtd_ativos
        w3['C' + str(first_empty_row)] = total_ativados
        w3['D' + str(first_empty_row)] = total_cancelados
        



W4 - RESUMO

In [29]:
import datetime 
# Garante que yesterday esteja normalizado
if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

# Hoje
hoje = datetime.date.today()
dia_semana = hoje.weekday() 

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    sexta_str = sexta.strftime('%d/%m/%Y')
    domingo_str = domingo.strftime('%d/%m/%Y')
    resumo_periodo = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    w4['C2'] = resumo_periodo
else:
    w4['C2'] = yest_date.strftime('%d/%m/%Y')

w4['C3'] = qtd_ativos

w4['C8'] = ativados_seg
w4['C9'] = ativados_st
w4['C10'] = ativados_viav
w4['C11'] = ativados_tag
total_ativados = ativados_seg + ativados_st + ativados_viav + ativados_tag
w4['C12'] = total_ativados


w4['D8'] = finalizados_seg_int
w4['D9'] = finalizados_st_int
w4['D10'] = finalizados_viav_int
w4['D11'] = finalizados_tag_int
total_finalizados = finalizados_seg_int + finalizados_st_int + finalizados_viav_int + finalizados_tag_int
w4['D12'] = total_finalizados

w4['E8'] = cancelados_seg_int
w4['E9'] = cancelados_st_int
w4['E10'] = cancelados_viav_int
w4['E11'] = cancelados_tag_int
total_cancelados = cancelados_seg_int + cancelados_st_int + cancelados_viav_int + cancelados_tag_int
w4['E12'] = total_cancelados

w4['F8'] = cancelados_seg_parc
w4['F9'] = cancelados_st_parc
w4['F10'] = cancelados_viav_parc
w4['F11'] = cancelados_tag_parc
total_beneficios = cancelados_seg_parc + cancelados_st_parc + cancelados_viav_parc + cancelados_tag_parc
w4['F12'] = total_beneficios

In [41]:
#gerando imagem

yesterday = datetime.date.today() - datetime.timedelta(days=1)

if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

hoje = datetime.date.today()
dia_semana = hoje.weekday() 

if dia_semana == 0:  # Segunda-feira
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    resumo_periodo = f"{sexta.strftime('%d/%m/%Y')} (sexta) - {domingo.strftime('%d/%m/%Y')} (domingo)"
else:
    resumo_periodo = yest_date.strftime('%d/%m/%Y')

# Adicionando separador de milhar em qtd_ativos
qtd_ativos_fmt = f"{qtd_ativos:,}".replace(",", ".")

html_content = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    body {{
        font-family: 'Calibri', 'Segoe UI', sans-serif;
        font-size: 14px;
        background-color: white;
        margin: 10px;
    }}
    table {{
        border-collapse: collapse;
        width: 700px;
        text-align: center;
    }}
    th, td {{
        border: 1px solid black;
        padding: 4px 8px;
    }}
    .bg-gray {{ background-color: #D9D9D9; font-weight: bold; }}
    .bg-orange {{ background-color: #FCE4D6; font-weight: bold; }}
    .bg-yellow {{ background-color: #FFF2CC; font-weight: bold; }}
    .bold {{ font-weight: bold; }}
    
    /* Ajustes específicos de largura para espelhar a imagem */
    .col-empresa {{ width: 15%; text-align: center; }}
    .col-ativadas {{ width: 18%; }}
    .col-fin {{ width: 22%; }}
    .col-canc {{ width: 22%; }}
    .col-ben {{ width: 23%; }}
</style>
</head>
<body>

    <table>
        <tr>
            <td class="bg-gray col-empresa">DATA</td>
            <td colspan="4" class="bold">{resumo_periodo}</td>
        </tr>
        <tr>
            <td class="bg-gray">TOTAL ATIVOS</td>
            <td colspan="4" class="bold">{qtd_ativos_fmt}</td>
        </tr>
    </table>

    <br>

    <table>
        <tr>
            <td rowspan="3" class="bold bg-gray col-empresa">EMPRESA</td>
            <td rowspan="3" class="bold bg-gray col-ativadas">PLACAS ATIVADAS</td>
            <td colspan="3" class="bold bg-gray">PLACAS CANCELADAS</td>
        </tr>
        <tr>
            <td colspan="2" class="bg-orange">CONJUNTOS CANCELADOS</td>
            <td rowspan="2" class="bg-yellow">BENEFÍCIOS CANCELADOS</td>
        </tr>
        <tr>
            <td class="bg-orange col-fin">FINALIZADOS</td>
            <td class="bg-orange col-canc">CANCELADOS</td>
        </tr>
        
        <tr>
            <td class="bg-gray">Segtruck</td>
            <td>{ativados_seg}</td>
            <td>{finalizados_seg_int}</td>
            <td>{cancelados_seg_int}</td>
            <td>{cancelados_seg_parc}</td>
        </tr>
        <tr>
            <td class="bg-gray">Stcoop</td>
            <td>{ativados_st}</td>
            <td>{finalizados_st_int}</td>
            <td>{cancelados_st_int}</td>
            <td>{cancelados_st_parc}</td>
        </tr>
        <tr>
            <td class="bg-gray">Viavante</td>
            <td>{ativados_viav}</td>
            <td>{finalizados_viav_int}</td>
            <td>{cancelados_viav_int}</td>
            <td>{cancelados_viav_parc}</td>
        </tr>
        <tr>
            <td class="bg-gray">Tag</td>
            <td>{ativados_tag}</td>
            <td>{finalizados_tag_int}</td>
            <td>{cancelados_tag_int}</td>
            <td>{cancelados_tag_parc}</td>
        </tr>
        
        <tr>
            <td class="bold bg-gray">Total</td>
            <td class="bold">{total_ativados}</td>
            <td class="bold">{total_finalizados}</td>
            <td class="bold">{total_cancelados}</td>
            <td class="bold">{total_beneficios}</td>
        </tr>
    </table>

</body>
</html>
"""

# =====================================================================
# 4. GERAÇÃO E EXPORTAÇÃO DA IMAGEM
# =====================================================================
caminho_saida = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
nome_arquivo = "relatorio_tabela.png"

# Garante que o diretório existe
os.makedirs(caminho_saida, exist_ok=True)

# Configura o renderizador de imagem (tamanho adaptado para a tabela)
hti = Html2Image(output_path=caminho_saida, size=(750, 350))

# Gera a imagem
hti.screenshot(html_str=html_content, save_as=nome_arquivo)

print(f"Sucesso! Imagem gerada em: {os.path.join(caminho_saida, nome_arquivo)}")

Sucesso! Imagem gerada em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\relatorio_tabela.png


### automação envio whatsapp

In [ ]:
time.sleep(2)
pyautogui.hotkey('win', 'e')
time.sleep(4)
pyautogui.hotkey('ctrl', 'l') 
time.sleep(1.5)
pyautogui.typewrite(r'C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'f') 
time.sleep(2.5)
pyautogui.typewrite(r'relatorio')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.press('down')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'c')
#whatsapp - primeiro arquivo
time.sleep(1.5)
pyautogui.press('win')
time.sleep(1.5)
pyautogui.typewrite('whatsapp')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(5)
pyautogui.hotkey('ctrl', 'f') 
time.sleep(1.5)
pyautogui.typewrite("ativ")
time.sleep(1.5)
pyautogui.press('down')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'v')
time.sleep(2.5)
pyautogui.typewrite(rf'Bom dia, pessoal! Segue o resultado das placas ativadas e canceladas nas 4 empresas. Refere-se ao(s) dia(s): {resumo_periodo}', interval=0.1)
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
#exclui arquivo
pyautogui.hotkey('alt', 'tab')
time.sleep(1.5)
pyautogui.press('del')

######################################################whatsapp - segundo arquivo
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'a')
# time.sleep(2.5)
# pyautogui.typewrite(r'tabela')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# time.sleep(1.5)
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')


### carregando resultados em excel

In [ ]:
# save workbook base template
wb.save(template)

# create historic backup
name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)


wb.close()